In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

In [ ]:
# 划分数据集
block_size = 3
def build_dataset(words):
    X, Y = [], []

    for w in words:
        chs = list(w) + ['.']
        context = [0] * block_size
        for ch in chs:
            ix = stoi[ch]

            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '---->', itos[ix])
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

**now made respectful**

In [ ]:
g = torch.Generator().manual_seed(1000) 
C = torch.randn((27, 10),generator= g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)
print(3*C.shape[1])
stepi = []
lossi = []

In [ ]:
for i in range(50000):
    #minibatch
    ix = torch.randint(0, Xtr.shape[0], (32,))

    # forward pass
    emb = C[Xtr[ix]]
    print(emb.shape)

    h = torch.tanh(emb.view(emb.shape[0],3*C.shape[1]) @ W1 + b1)

    logit = h @ W2 + b2
    # counts = logit.exp()
    # probs = counts / counts.sum(1,keepdim=True)
    # lossprob = -probs[torch.arange(32),Y].log().mean()
    loss = F.cross_entropy(logit, Ytr[ix])
    #print(f'loss = {loss.item()}')

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.01

    for p in parameters:
        p.data += -lr * p.grad

    stepi.append(i) 
    lossi.append(loss.log10().item())

print(f'loss = {loss.item()}')

In [ ]:
emb = C[Xdev]
# print(emb.shape)
h = torch.tanh(emb.view(emb.shape[0], 3*C.shape[1]) @ W1 + b1)
logit = h @ W2 + b2
loss = F.cross_entropy(logit, Ydev)
loss

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
# 采样
g = torch.Generator().manual_seed(2001)
block_size = 3
for _ in range(20):
    context = [0] * block_size
    out = []
    while True:
        emb = C[torch.tensor(context)]
        # print(emb.shape, C.shape, W1.shape)
        h = torch.tanh(emb.view(-1, block_size * C.shape[1]) @ W1 + b1)
        logit = h @ W2 + b2
        counts = logit.exp()
        probs = counts / counts.sum(dim=1, keepdim=True)
        ix = torch.multinomial(probs, num_samples=1, generator=g)

        ch = itos[ix.item()]
        context = context[1:] + [ix]
        out.append(ch)
        if ix == 0:
            break
    print(''.join(i for i in out))